# 07 — Monthly Predictor Stack

Builds monthly multiband raster stacks from aligned predictors.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import re
import rasterio

aligned_predictors = PROCESSED_DIR / "aligned_rasters" / "predictors"
stack_dir = PROCESSED_DIR / "predictor_stack"
stack_dir.mkdir(parents=True, exist_ok=True)

static_names = ["DEM", "Slope", "Aspect", "Distance_Sea"]
static_files = {}
for name in static_names:
    matches = sorted((aligned_predictors / name).glob("*.tif"))
    if matches:
        static_files[name] = matches[0]

def find_monthly_file(folder: Path, year: int, month: int):
    patterns = [
        f"*{year}_{month}.tif",
        f"*{year}_{month:02d}.tif",
    ]
    for pattern in patterns:
        matches = sorted(folder.glob(pattern))
        if matches:
            return matches[0]
    return None

In [ ]:
for year in range(2017, 2023):
    for month in range(1, 13):
        layers = []
        names = []

        ndvi = find_monthly_file(aligned_predictors / "NDVI", year, month)
        lst = find_monthly_file(aligned_predictors / "LST_Day", year, month)

        dynamic = {"NDVI": ndvi, "LST_Day": lst}
        selected = {**static_files, **dynamic}
        selected = {k: v for k, v in selected.items() if v is not None}

        if not selected:
            continue

        reference = next(iter(selected.values()))
        with rasterio.open(reference) as ref:
            profile = ref.profile.copy()
            profile.update(count=len(selected), compress="lzw")

            output = stack_dir / f"predictor_stack_{year}_{month:02d}.tif"
            with rasterio.open(output, "w", **profile) as dst:
                for band_index, (name, path) in enumerate(selected.items(), start=1):
                    with rasterio.open(path) as src:
                        dst.write(src.read(1), band_index)
                    dst.set_band_description(band_index, name)

print("Monthly predictor stacks created.")